# LangChain RAG Demo

LangChain with LCEL (LangChain Expression Language) for composable RAG pipelines.

**Why LangChain:**
- Largest ecosystem of integrations
- LCEL provides clean composition syntax
- Good for rapid prototyping

**Prerequisites:**
```bash
pip install langchain langchain-community langchain-core sentence-transformers faiss-cpu
ollama pull qwen3:4b
```

In [1]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import requests

# Sample documents
documents = [
    Document(page_content="Remote work is allowed 3 days per week with manager approval.", metadata={"topic": "hr"}),
    Document(page_content="Vacation policy: 25 days annual leave for full-time employees.", metadata={"topic": "hr"}),
    Document(page_content="Code reviews are mandatory before merging to main branch.", metadata={"topic": "eng"}),
    Document(page_content="Performance reviews occur quarterly with written feedback.", metadata={"topic": "hr"}),
]

print(f"✓ {len(documents)} documents ready")

✓ 4 documents ready


---

## 1. Create Vector Store

In [2]:
# Create embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✓ Vector store created")

# Test retrieval
query = "How many vacation days?"
docs = retriever.invoke(query)
print(f"\nQuery: \"{query}\"")
for doc in docs:
    print(f"  - {doc.page_content}")

/var/folders/7r/97vw1q7x18x8btt349jvjpvc0000gn/T/ipykernel_293/2867722242.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


✓ Vector store created

Query: "How many vacation days?"
  - Vacation policy: 25 days annual leave for full-time employees.
  - Remote work is allowed 3 days per week with manager approval.
  - Performance reviews occur quarterly with written feedback.


---

## 2. Create Ollama LLM Wrapper

In [3]:
from langchain_core.language_models.llms import LLM
from typing import Any, List, Optional

class OllamaLLM(LLM):
    model: str = "qwen3:4b"
    
    @property
    def _llm_type(self) -> str:
        return "ollama"
    
    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": self.model, "prompt": prompt, "stream": False}
        )
        return response.json().get("response", "")

llm = OllamaLLM()
print("✓ Ollama LLM wrapper created")

✓ Ollama LLM wrapper created


---

## 3. LCEL RAG Chain

In [4]:
# LCEL-style RAG chain
prompt = ChatPromptTemplate.from_template("""Answer based only on the context. No thinking, just answer.

Context:
{context}

Question: {question}
Answer:""")

def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

# LCEL composition with pipe operator
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ LCEL RAG chain built")

✓ LCEL RAG chain built


In [5]:
# Run the chain
try:
    query = "What is the vacation policy?"
    result = rag_chain.invoke(query)
    
    print(f"Query: \"{query}\"")
    print("=" * 60)
    print(f"Answer: {result[:300]}...")
except Exception as e:
    print(f"Chain error: {e}")
    print("Make sure Ollama is running with qwen3:4b")

Query: "What is the vacation policy?"
Answer: Answer: 25 days annual leave for full-time employees....


---

## LCEL Key Concepts

**Pipe Operator (`|`):** Chains components where output of left feeds into right  
**RunnablePassthrough:** Passes input through unchanged  
**RunnableLambda:** Wraps any function as a chain component

**LCEL vs Legacy Chains:**
```python
# Legacy (deprecated)
chain = RetrievalQA.from_chain_type(llm, retriever=retriever)

# LCEL (current)
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)
```

**When to use LangChain:** Rapid prototyping, many integrations needed  
**When to use Haystack:** Type-safety important, production deployment